# Exp 13: Oversampled views

Card: [exp13_oversampled_views.md](../../docs/neuroslope/experiments/exp13_oversampled_views.md)


In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from dataset.augment import Validation
from training.config import ROOT, SEED, TRAIN_DIR
from training.config import VIP
from training.runs import read_config
from training.cli import data_module
from analysis.plots import to_image, save_figures, use_style

N_IMAGES = 8
MAX_DRAWS = None  # all draws
CONFIG = ROOT / "configs" / "models" / "exp11_resnet18_5fold" / "resnet18.yaml"
FOLD = 0
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp13_oversampled_views"

use_style()

## One epoch of sampler


In [ ]:
torch.manual_seed(SEED)
dm = data_module(CONFIG, FOLD)
dm.setup()
train = dm.train_ds.df
draws = Counter(dm.train_dataloader().sampler)  # dataset row: times drawn in epoch
vip = train[VIP].any(axis=1)
print(
    f"draws per image, mean: VIP {np.mean([draws[i] for i in train.index[vip]]):.2f}, "
    f"other {np.mean([draws[i] for i in train.index[~vip]]):.2f}"
)

## Views per VIP class


In [ ]:
resize = Validation(dm.hparams.img_size).compose()
rng = np.random.default_rng(SEED)
for cls in VIP:
    rows = rng.choice(train.index[train[cls] == 1], N_IMAGES, replace=False)
    n_cols = 1 + (MAX_DRAWS or max(draws[i] for i in rows))
    fig, axes = plt.subplots(
        N_IMAGES, n_cols, figsize=(1.8 * n_cols, 2.0 * N_IMAGES), squeeze=False
    )
    for ax_row, i in zip(axes, rows):
        image = np.array(
            Image.open(TRAIN_DIR / f"{train.loc[i, 'image_id']}.jpg").convert("RGB")
        )
        ax_row[0].imshow(to_image(resize(image=image)["image"]))
        ax_row[0].set_title(f"original · {draws[i]} draws", fontsize=8)
        for ax in ax_row[1 : 1 + min(draws[i], n_cols - 1)]:
            ax.imshow(
                to_image(dm.train_ds[i][0])
            )  # fresh augmentation per draw, as in training
        for ax in ax_row:
            ax.set_axis_off()
    fig.suptitle(
        f"{cls} · fold {FOLD} train · one epoch, oversample {dm.oversample}, augment {read_config(CONFIG)['data'].get('augment')}"
    )
    fig.tight_layout()
    save_figures({f"{PREFIX}_{cls.lower()}.png": fig}, FIGURES, dpi=100)
    plt.to_image()